# OpenAI Realtime + LangSmith

Build the speech-to-speech agent in the notebook. The hidden pieces are only local mic/speaker setup and mic-frame pumping.

In [ ]:
import asyncio
import base64
import json
import os
import uuid

from dotenv import load_dotenv
from openai import AsyncOpenAI
from langsmith.integrations.openai_realtime import wrap_realtime

from voice_demo.openai.utils import execute_tool
from voice_demo.workshop import openai_console_io, pump_openai_mic

load_dotenv()

PROJECT = "voice-workshop-s2s"
MODEL = os.getenv("REALTIME_MODEL", "gpt-realtime-2")
SAMPLE_RATE = 24_000

assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running this notebook."
assert os.getenv("LANGSMITH_API_KEY"), "Set LANGSMITH_API_KEY before running this notebook."

In [ ]:
SYSTEM_PROMPT = """You are a friendly voice assistant who can look up the
weather for any city. Keep replies short, conversational, and free of
formatting. When the user asks about weather, call lookup_weather once per
city, then summarize the result in one or two spoken sentences."""

WEATHER_TOOL = {
    "type": "function",
    "name": "lookup_weather",
    "description": "Get the current weather for a single city. Call once per city.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "City name, e.g. Paris."}
        },
        "required": ["city"],
    },
}

In [ ]:
SESSION = {
    "type": "realtime",
    "instructions": SYSTEM_PROMPT,
    "output_modalities": ["audio"],
    "audio": {
        "input": {
            "format": {"type": "audio/pcm", "rate": SAMPLE_RATE},
            "transcription": {"model": "gpt-4o-mini-transcribe"},
            "noise_reduction": {"type": "near_field"},
            "turn_detection": {
                "type": "server_vad",
                "create_response": False,
                "interrupt_response": True,
            },
        },
        "output": {
            "format": {"type": "audio/pcm", "rate": SAMPLE_RATE},
            "voice": "alloy",
        },
    },
    "tools": [WEATHER_TOOL],
    "tool_choice": "auto",
}

In [ ]:
async def handle_realtime_event(connection, event, audio_out, ui) -> None:
    if event.type == "response.output_audio.delta":
        audio_out.write(base64.b64decode(event.delta))

    elif event.type == "input_audio_buffer.speech_started":
        audio_out.clear()

    elif event.type == "conversation.item.input_audio_transcription.completed":
        transcript = (event.transcript or "").strip()
        if transcript:
            ui.log(f"user:  {transcript}")
            await connection.response.create()

    elif event.type == "response.output_audio_transcript.done":
        transcript = (event.transcript or "").strip()
        if transcript:
            ui.log(f"agent: {transcript}")

    elif event.type == "response.done":
        tool_calls = [
            item for item in (event.response.output or [])
            if item.type == "function_call"
        ]
        for call in tool_calls:
            result = await execute_tool(call.name, call.arguments)
            await connection.conversation.item.create(
                item={
                    "type": "function_call_output",
                    "call_id": call.call_id,
                    "output": json.dumps(result),
                }
            )
        if tool_calls:
            await connection.response.create()

    elif event.type == "error":
        raise RuntimeError(event.error)

In [ ]:
# Uses your local mic and speaker. Stop/cancel this cell to end the session cleanly.
client = AsyncOpenAI()
audio_in, audio_out, ui = openai_console_io(SAMPLE_RATE)
thread_id = str(uuid.uuid4())
mic_task = None

async with client.realtime.connect(model=MODEL) as raw, wrap_realtime(
    raw,
    thread_id=thread_id,
    sample_rate=SAMPLE_RATE,
    project_name=PROJECT,
    tags=["workshop", "openai-realtime"],
    metadata={"model": MODEL},
    is_agent_speaking=lambda: audio_out.buffered_bytes() > 0,
) as connection:
    await connection.session.update(session=SESSION)
    audio_out.set_played_callback(connection.record_agent_audio)
    audio_in.start()
    audio_out.start()
    ui.log(f"[openai] thread_id={thread_id}")
    ui.log("[openai] connected. Stop/cancel this cell to end cleanly.")
    mic_task = asyncio.create_task(pump_openai_mic(connection, audio_in, ui))

    try:
        async for event in connection:
            await handle_realtime_event(connection, event, audio_out, ui)
    except asyncio.CancelledError:
        task = asyncio.current_task()
        while task is not None and task.cancelling():
            task.uncancel()
        await connection.close(code=1000, reason="notebook cell stopped")
    finally:
        if mic_task is not None:
            mic_task.cancel()
            await asyncio.gather(mic_task, return_exceptions=True)
        audio_in.stop()
        audio_out.stop()
        ui.finish()